# Customer data analysis using Plotly

In [15]:
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv


In [16]:

load_dotenv()
db_name = os.getenv('DB_NAME')
db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')

# Create SQLAlchemy engine
engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

## Read customer data from database

In [17]:
# Read customers table from the database
query = 'SELECT * FROM customers'
df_customers = pd.read_sql_query(query, engine)

print("Customers:")
display(df_customers.head())

Customers:


,customer_id,first_name,last_name,email,city,region,country,signup_date,age,is_subscribed,loyalty_tier
0,CUST-0001,James,Brown,james.brown@gmail.com,London,England,United Kingdom,2022-01-31,65,True,Bronze
1,CUST-0002,Emily,Campbell,emily.campbell@outlook.com,Birmingham,England,United Kingdom,2023-09-25,55,True,Bronze
2,CUST-0003,Arthur,Roberts,arthur.roberts@icloud.com,London,England,United Kingdom,2021-03-25,53,False,Bronze
3,CUST-0004,Freddie,Murphy,freddie.murphy@outlook.com,London,England,United Kingdom,2020-10-15,66,False,Bronze
4,CUST-0005,Leo,Stewart,leo.stewart@hotmail.com,Glasgow,Scotland,United Kingdom,2020-01-15,24,False,Bronze


### Bar chart of total customers by region

In [18]:

region_counts = df_customers['region'].value_counts().sort_index()
fig1 = px.bar(
    x=region_counts.index,
    y=region_counts.values,
    labels={'x': 'Region', 'y': 'Total Customers'},
    title='Total Customers by Region',
    text=region_counts.values,
    color=region_counts.index,
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig1.update_traces(texttemplate='%{text}', textposition='inside', textfont_color='white')
fig1.update_layout(
    xaxis_title='',
    yaxis_title='',
    uniformtext_minsize=8,
    uniformtext_mode='hide',
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
    )
fig1.show()


### Grouped bar chart of age distribution by region

In [19]:

bins = [18, 30, 50, 100]
labels = ["18-29", "30-49", "50+"]

df_customers['age_group'] = pd.cut(
    df_customers['age'], bins=bins, labels=labels, right=False
    )
age_region_counts = df_customers.groupby(['region', 'age_group']).size().reset_index(name='count')
fig2 = px.bar(
    age_region_counts,
    x='region',
    y='count',
    color='age_group',
    barmode='group',
    labels={'region': 'Region', 'count': 'Customer Count', 'age_group': 'Age Range'},
    title='Age Distribution of Customers by Region',
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig2.update_layout(
    xaxis_title='Region',
    yaxis_title='Customer Count',
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
    )
fig2.show()


### Pie chart for overall age distribution of customers


In [20]:

age_counts = df_customers['age_group'].value_counts().sort_index()
fig3 = px.pie(
    values=age_counts.values,
    names=age_counts.index,
    title='Age Distribution of All Customers',
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig3.update_traces(textinfo='percent+label', textfont_color='white')
fig3.update_layout(
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
    )
fig3.show()



### Grouped bar chart of age range vs loyalty tier and region (percentage)


In [21]:

tier_order = ['Bronze', 'Silver', 'Gold', 'Platinum']
tier_colors = {'Bronze': '#cd7f32', 'Silver': '#c0c0c0', 'Gold': '#ffd700', 'Platinum': '#e5e4e2'}
tier_counts = df_customers.groupby(['region', 'age_group', 'loyalty_tier']).size().reset_index(name='count')
total_by_age_region = df_customers.groupby(['region', 'age_group']).size().reset_index(name='total')
merged = pd.merge(tier_counts, total_by_age_region, on=['region', 'age_group'])
merged['percent'] = merged['count'] / merged['total'] * 100
merged['loyalty_tier'] = pd.Categorical(merged['loyalty_tier'], categories=tier_order, ordered=True)
fig4 = px.bar(
    merged.sort_values('loyalty_tier'),
    x='region',
    y='percent',
    color='loyalty_tier',
    facet_col='age_group',
    barmode='stack',
    labels={'region': 'Region', 'percent': 'Percent of Loyalty Tier', 'loyalty_tier': 'Loyalty Tier', 'age_group': 'Age Range'},
    title='Percentage of Loyalty Tier in Each Age Group by Region',
    color_discrete_map=tier_colors
    )
fig4.update_layout(
    xaxis_title='',
    yaxis_title='Percent',
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
    )
fig4.show()
